# Computer Vision Nanodegree

## Project: Image Captioning

---

In this notebook, you will train your CNN-RNN model.  

You are welcome and encouraged to try out many different architectures and hyperparameters when searching for a good model.

This does have the potential to make the project quite messy!  Before submitting your project, make sure that you clean up:
- the code you write in this notebook.  The notebook should describe how to train a single CNN-RNN architecture, corresponding to your final choice of hyperparameters.  You should structure the notebook so that the reviewer can replicate your results by running the code in this notebook.  
- the output of the code cell in **Step 2**.  The output should show the output obtained when training the model from scratch.

This notebook **will be graded**.  

Feel free to use the links below to navigate the notebook:
- [Step 1](#step1): Training Setup
- [Step 2](#step2): Train your Model
- [Step 3](#step3): (Optional) Validate your Model

<a id='step1'></a>
## Step 1: Training Setup

In this step of the notebook, you will customize the training of your CNN-RNN model by specifying hyperparameters and setting other options that are important to the training procedure.  The values you set now will be used when training your model in **Step 2** below.

You should only amend blocks of code that are preceded by a `TODO` statement.  **Any code blocks that are not preceded by a `TODO` statement should not be modified**.

### Task #1

Begin by setting the following variables:
- `batch_size` - the batch size of each training batch.  It is the number of image-caption pairs used to amend the model weights in each training step. 
- `vocab_threshold` - the minimum word count threshold.  Note that a larger threshold will result in a smaller vocabulary, whereas a smaller threshold will include rarer words and result in a larger vocabulary.  
- `vocab_from_file` - a Boolean that decides whether to load the vocabulary from file. 
- `embed_size` - the dimensionality of the image and word embeddings.  
- `hidden_size` - the number of features in the hidden state of the RNN decoder.  
- `num_epochs` - the number of epochs to train the model.  We recommend that you set `num_epochs=3`, but feel free to increase or decrease this number as you wish.  [This paper](https://arxiv.org/pdf/1502.03044.pdf) trained a captioning model on a single state-of-the-art GPU for 3 days, but you'll soon see that you can get reasonable results in a matter of a few hours!  (_But of course, if you want your model to compete with current research, you will have to train for much longer._)
- `save_every` - determines how often to save the model weights.  We recommend that you set `save_every=1`, to save the model weights after each epoch.  This way, after the `i`th epoch, the encoder and decoder weights will be saved in the `models/` folder as `encoder-i.pkl` and `decoder-i.pkl`, respectively.
- `print_every` - determines how often to print the batch loss to the Jupyter notebook while training.  Note that you **will not** observe a monotonic decrease in the loss function while training - this is perfectly fine and completely expected!  You are encouraged to keep this at its default value of `100` to avoid clogging the notebook, but feel free to change it.
- `log_file` - the name of the text file containing - for every step - how the loss and perplexity evolved during training.

If you're not sure where to begin to set some of the values above, you can peruse [this paper](https://arxiv.org/pdf/1502.03044.pdf) and [this paper](https://arxiv.org/pdf/1411.4555.pdf) for useful guidance!  **To avoid spending too long on this notebook**, you are encouraged to consult these suggested research papers to obtain a strong initial guess for which hyperparameters are likely to work best.  Then, train a single model, and proceed to the next notebook (**3_Inference.ipynb**).  If you are unhappy with your performance, you can return to this notebook to tweak the hyperparameters (and/or the architecture in **model.py**) and re-train your model.

### Question 1

**Question:** Describe your CNN-RNN architecture in detail.  With this architecture in mind, how did you select the values of the variables in Task 1?  If you consulted a research paper detailing a successful implementation of an image captioning model, please provide the reference.

**Answer:**
The architecture is a standard **CNN-encoder + LSTM-decoder** caption model.

* **Encoder** — a pretrained ResNet-18 with the final classification
  layer removed; its 512-dim feature vector is projected to
  `embed_size` via a `nn.Linear` and passed through a 1-D BatchNorm.
  All ResNet weights are frozen so only the projection layer is
  trained, which keeps the GPU footprint small and reuses ImageNet's
  visual features (which are well-suited to natural images).
* **Decoder** — a 1-layer `nn.LSTM` whose vocabulary is embedded by
  an `nn.Embedding(vocab_size, embed_size)`. The image embedding is
  fed in as the **first time step** (à la Vinyals et al. *Show and
  Tell*), then the caption tokens (with `<end>` dropped) follow with
  teacher forcing. The LSTM hidden state goes through `nn.Linear` to
  produce per-token logits over the vocabulary.

I selected the Task-1 hyperparameters with two priorities: **stay
close to the original *Show and Tell* paper** and **fit comfortably in
the workspace's GPU memory**.

* `batch_size = 32` — small enough to fit ResNet activations + LSTM
  state on the workspace GPU comfortably; large enough that the
  per-step gradient is not too noisy. (The training loop also uses 4
  gradient-accumulation steps, so the *effective* batch is 128.)
* `vocab_threshold = 5` — same threshold the project notebook
  recommends and the value used in the Karpathy splits and *Show and
  Tell* paper. Below 5, rare words bloat the vocabulary; above 5,
  too many useful words become `<unk>`.
* `vocab_from_file = True` — once the vocab pickle has been built
  once, loading it from disk is dramatically faster than re-tokenising
  the whole corpus on every notebook restart.
* `embed_size = 256` and `hidden_size = 512` — the embedding /
  hidden ratio reported in *Show and Tell* (Vinyals et al., 2015).
  256 is small enough to keep the LSTM lightweight and 512 is large
  enough to capture the per-step state.
* `num_epochs = 3` — the project explicitly recommends this for the
  small COCO subset; with the frozen ResNet most of the trainable
  parameters are in the LSTM and they converge quickly.

**Reference:** Vinyals, Toshev, Bengio, Erhan — *Show and Tell: A
Neural Image Caption Generator*, arXiv:1411.4555 (2015).


### (Optional) Task #2

Note that we have provided a recommended image transform `transform_train` for pre-processing the training images, but you are welcome (and encouraged!) to modify it as you wish.  When modifying this transform, keep in mind that:
- the images in the dataset have varying heights and widths, and 
- if using a pre-trained model, you must perform the corresponding appropriate normalization.

### Question 2

**Question:** How did you select the transform in `transform_train`?  If you left the transform at its provided value, why do you think that it is a good choice for your CNN architecture?

**Answer:**
I kept `transform_train` essentially as provided. The pipeline is
`Resize(256) → RandomCrop(224) → ToTensor → Normalize(ImageNet
mean/std)`, which is the **canonical preprocessing pipeline for any
ImageNet-pretrained ResNet** — including ResNet-18 in the encoder.

Three reasons that pipeline is correct here:

1. **Spatial size matches ResNet-18.** ResNet-18 was trained on
   224×224 inputs, so the encoder expects 224×224 tensors. Resizing
   the smaller edge to 256 and then random-cropping to 224 keeps the
   aspect ratio (no squashing) and gives the network a slight bit of
   spatial augmentation between epochs.
2. **Normalisation must match the pretrained weights.** ResNet-18's
   pretrained weights expect inputs whose channels are normalised
   with `mean=(0.485, 0.456, 0.406)` and
   `std=(0.229, 0.224, 0.225)`. Skipping or changing this would
   silently shift the input distribution and degrade the encoder's
   features.
3. **Random cropping is a cheap augmentation that doesn't risk
   distorting captions.** Unlike colour jitter or horizontal flip
   (which would invert "the man on the *left*" type captions),
   random cropping just changes which 224×224 patch the network
   sees, which improves generalisation without breaking the
   image–caption alignment.

### Task #3

Next, you will specify a Python list containing the learnable parameters of the model.  For instance, if you decide to make all weights in the decoder trainable, but only want to train the weights in the embedding layer of the encoder, then you should set `params` to something like:
```
params = list(decoder.parameters()) + list(encoder.embed.parameters()) 
```

### Question 3

**Question:** How did you select the trainable parameters of your architecture?  Why do you think this is a good choice?

**Answer:**
I set
```python
params = list(decoder.parameters()) + list(encoder.embed.parameters())
```
That is, **the entire LSTM decoder is trainable**, plus **only the
linear projection layer of the encoder** (`encoder.embed`). The
ResNet-18 backbone and the BatchNorm are frozen.

Why this is a good split:

* The ResNet-18 backbone was pretrained on ImageNet and already
  produces strong general visual features. Fine-tuning it on the
  small ~2000-image COCO subset would risk **catastrophic
  forgetting** with no real upside, because the pretrained features
  are good enough for caption-relevant objects (dogs, people,
  vehicles, food, etc.).
* The single trainable layer in the encoder is the
  `nn.Linear(512 → embed_size)` projection — that one layer is
  what *adapts* ImageNet features into the caption embedding space,
  so it absolutely needs to be trained.
* Every parameter in the decoder is new and randomly initialised
  (the word embedding, the LSTM, and the output linear layer), so
  it all has to be trained.

This is the same scheme used in *Show and Tell* and matches what
most published image-captioning baselines do.

### Task #4

Finally, you will select an [optimizer](http://pytorch.org/docs/master/optim.html#torch.optim.Optimizer).

### Question 4

**Question:** How did you select the optimizer used to train your model?

**Answer:**
I chose **Adam** (`torch.optim.Adam`) with a learning rate of
`1e-3` and default `betas=(0.9, 0.999)`.

Why Adam over plain SGD here:

* **Adaptive per-parameter learning rates.** The trainable parameter
  set is heterogeneous — a small linear projection in the encoder, a
  large word-embedding matrix, an LSTM, and a final linear layer.
  These will have very different gradient magnitudes; Adam's
  per-parameter scaling handles that without manual tuning.
* **No need to schedule the learning rate to get reasonable
  results.** With only 3 epochs over a 2000-image subset, an SGD +
  scheduler combo would be over-engineered for the time budget.
* **Standard for sequence models.** Adam (or AdamW) is the de-facto
  optimiser for LSTM/Transformer-style language models, including
  *Show and Tell*'s replication papers.
* **Default `lr=1e-3` is a sensible starting point** for word
  embeddings + an LSTM on top of pretrained image features. If
  perplexity plateaued I would lower it to `5e-4`, but for a 3-epoch
  run `1e-3` converges quickly without divergence. 

In [1]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/yousefradwan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
import torch
import torch.nn as nn
from torchvision import transforms
import sys
sys.path.append('/tmp/coco/cocoapi/PythonAPI')
from pycocotools.coco import COCO
from data_loader import get_loader
from model import EncoderCNN, DecoderRNN
import math


## TODO #1: Select appropriate values for the Python variables below.
batch_size = 32                        # batch size
vocab_threshold = 5                    # minimum word count threshold (Show and Tell uses 5)
vocab_from_file = True                 # load existing vocab.pkl after first run for speed
embed_size = 256                       # dimensionality of image and word embeddings
hidden_size = 512                      # number of features in the LSTM hidden state
num_epochs = 3                         # number of training epochs
save_every = 1                         # save model weights every epoch
print_every = 100                      # print loss every 100 steps
log_file = 'training_log.txt'          # log file with per-step loss / perplexity

# (Optional) TODO #2: Amend the image transform below.
# Standard ImageNet preprocessing for a pretrained ResNet backbone:
# resize the smaller edge to 256, take a random 224x224 crop, normalise
# with ImageNet channel means/stds. Random crop gives mild augmentation
# without breaking image-caption alignment (no flip/jitter).
transform_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225))])

# Build data loader (subset_size kept modest to fit in workspace time budget)
data_loader = get_loader(transform=transform_train,
                         mode='train',
                         batch_size=batch_size,
                         vocab_threshold=vocab_threshold,
                         vocab_from_file=vocab_from_file,
                         subset_size=2000)

# The size of the vocabulary.
vocab_size = len(data_loader.dataset.vocab)

# Initialize the encoder and decoder.
encoder = EncoderCNN(embed_size)
decoder = DecoderRNN(embed_size, hidden_size, vocab_size)

# Move models to GPU if CUDA is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder.to(device)
decoder.to(device)

# Define the loss function.
criterion = nn.CrossEntropyLoss().cuda() if torch.cuda.is_available() else nn.CrossEntropyLoss()

# TODO #3: Specify the learnable parameters of the model.
# Train the entire decoder + only the linear projection of the encoder;
# the ResNet-18 backbone stays frozen (catastrophic-forgetting-safe and
# the pretrained ImageNet features are already good for COCO).
params = list(decoder.parameters()) + list(encoder.embed.parameters())

# TODO #4: Define the optimizer.
# Adam(lr=1e-3) — adaptive per-parameter LR is well-suited to the mixed
# parameter set (small linear, large embedding, LSTM, output linear) and
# is the de-facto choice in image-captioning baselines.
optimizer = torch.optim.Adam(params=params, lr=1e-3)

# Set the total number of training steps per epoch.
total_step = math.ceil(len(data_loader.dataset.caption_lengths) / data_loader.batch_sampler.batch_size)


Vocabulary successfully loaded from vocab.pkl file!
loading annotations into memory...


Done (t=0.23s)
creating index...
index created!
Obtaining caption lengths for the subset...


  0%|          | 0/2000 [00:00<?, ?it/s]

100%|██████████| 2000/2000 [00:00<00:00, 26974.14it/s]


/Users/yousefradwan/miniconda3/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/yousefradwan/miniconda3/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


<a id='step2'></a>
## Step 2: Train your Model

Once you have executed the code cell in **Step 1**, the training procedure below should run without issue.  

It is completely fine to leave the code cell below as-is without modifications to train your model.  However, if you would like to modify the code used to train the model below, you must ensure that your changes are easily parsed by your reviewer.  In other words, make sure to provide appropriate comments to describe how your code works!  

You may find it useful to load saved weights to resume training.  In that case, note the names of the files containing the encoder and decoder weights that you'd like to load (`encoder_file` and `decoder_file`).  Then you can load the weights by using the lines below:

```python
# Load pre-trained weights before resuming training.
encoder.load_state_dict(torch.load(os.path.join('./models', encoder_file)))
decoder.load_state_dict(torch.load(os.path.join('./models', decoder_file)))
```

While trying out parameters, make sure to take extensive notes and record the settings that you used in your various training runs.  In particular, you don't want to encounter a situation where you've trained a model for several hours but can't remember what settings you used :).

### A Note on Tuning Hyperparameters

To figure out how well your model is doing, you can look at how the training loss and perplexity evolve during training - and for the purposes of this project, you are encouraged to amend the hyperparameters based on this information.  

However, this will not tell you if your model is overfitting to the training data, and, unfortunately, overfitting is a problem that is commonly encountered when training image captioning models.  

For this project, you need not worry about overfitting. **This project does not have strict requirements regarding the performance of your model**, and you just need to demonstrate that your model has learned **_something_** when you generate captions on the test data.  For now, we strongly encourage you to train your model for the suggested 3 epochs without worrying about performance; then, you should immediately transition to the next notebook in the sequence (**3_Inference.ipynb**) to see how your model performs on the test data.  If your model needs to be changed, you can come back to this notebook, amend hyperparameters (if necessary), and re-train the model.

That said, if you would like to go above and beyond in this project, you can read about some approaches to minimizing overfitting in section 4.3.1 of [this paper](http://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=7505636).  In the next (optional) step of this notebook, we provide some guidance for assessing the performance on the validation dataset.

In [3]:
import torch.utils.data as data
import numpy as np
import os
import requests
import time

# Open the training log file.
f = open(log_file, 'w')

accumulation_steps = 4  # Define how many steps to accumulate gradients
optimizer.zero_grad()   # Initialize gradients to zero

for epoch in range(1, num_epochs+1):
    
    for i_step in range(1, total_step+1):
        
        # Randomly sample a caption length, and sample indices with that length.
        indices = data_loader.dataset.get_train_indices()
        # Create and assign a batch sampler to retrieve a batch with the sampled indices.
        new_sampler = data.sampler.SubsetRandomSampler(indices=indices)
        data_loader.batch_sampler.sampler = new_sampler
        
        # Obtain the batch.
        images, captions = next(iter(data_loader))

        # Move batch of images and captions to GPU if CUDA is available.
        images = images.to(device)
        captions = captions.to(device)
        
        # Pass the inputs through the CNN-RNN model.
        features = encoder(images)
        outputs = decoder(features, captions)
        
        # Calculate the batch loss.
        loss = criterion(outputs.view(-1, vocab_size), captions.view(-1))
        loss = loss / accumulation_steps  # Scale the loss by the accumulation steps

        # Backward pass.
        loss.backward()

        if i_step % accumulation_steps == 0 or i_step == total_step:
            # Update the parameters in the optimizer and zero the gradients.
            optimizer.step()
            optimizer.zero_grad()
            
        # Get training statistics.
        stats = 'Epoch [%d/%d], Step [%d/%d], Loss: %.4f, Perplexity: %5.4f' % (epoch, num_epochs, i_step, total_step, loss.item() * accumulation_steps, np.exp(loss.item() * accumulation_steps))
        
        # Print training statistics (on same line).
        print('\r' + stats, end="")
        sys.stdout.flush()
        
        # Print training statistics to file.
        f.write(stats + '\n')
        f.flush()
        
        # Print training statistics (on different line).
        if i_step % print_every == 0:
            print('\r' + stats)
            
    # Save the weights.
    if epoch % save_every == 0:
        torch.save(decoder.state_dict(), os.path.join('./models', 'decoder-%d.pkl' % epoch))
        torch.save(encoder.state_dict(), os.path.join('./models', 'encoder-%d.pkl' % epoch))

# Close the training log file.
f.close()

Epoch [1/3], Step [1/63], Loss: 9.2012, Perplexity: 9908.6207

Epoch [1/3], Step [2/63], Loss: 9.1929, Perplexity: 9827.5834

Epoch [1/3], Step [3/63], Loss: 9.1914, Perplexity: 9812.7114

Epoch [1/3], Step [4/63], Loss: 9.1956, Perplexity: 9854.1522

Epoch [1/3], Step [5/63], Loss: 9.0870, Perplexity: 8839.4325

...

Epoch [3/3], Step [59/63], Loss: 4.3153, Perplexity: 74.8329

Epoch [3/3], Step [60/63], Loss: 4.5851, Perplexity: 98.0158

Epoch [3/3], Step [61/63], Loss: 4.7017, Perplexity: 110.1358

Epoch [3/3], Step [62/63], Loss: 4.4081, Perplexity: 82.1155

Epoch [3/3], Step [63/63], Loss: 4.6974, Perplexity: 109.6607

<a id='step3'></a>
## Step 3: (Optional) Validate your Model

To assess potential overfitting, one approach is to assess performance on a validation set.  If you decide to do this **optional** task, you are required to first complete all of the steps in the next notebook in the sequence (**3_Inference.ipynb**); as part of that notebook, you will write and test code (specifically, the `sample` method in the `DecoderRNN` class) that uses your RNN decoder to generate captions.  That code will prove incredibly useful here. 

If you decide to validate your model, please do not edit the data loader in **data_loader.py**.  Instead, create a new file named **data_loader_val.py** containing the code for obtaining the data loader for the validation data.  You can access:
- the validation images at filepath `'/opt/cocoapi/images/train2014/'`, and
- the validation image caption annotation file at filepath `'/opt/cocoapi/annotations/captions_val2014.json'`.

The suggested approach to validating your model involves creating a json file such as [this one](https://github.com/cocodataset/cocoapi/blob/master/results/captions_val2014_fakecap_results.json) containing your model's predicted captions for the validation images.  Then, you can write your own script or use one that you [find online](https://github.com/tylin/coco-caption) to calculate the BLEU score of your model.  You can read more about the BLEU score, along with other evaluation metrics (such as TEOR and Cider) in section 4.1 of [this paper](https://arxiv.org/pdf/1411.4555.pdf).  For more information about how to use the annotation file, check out the [website](http://cocodataset.org/#download) for the COCO dataset.

In [4]:
# (Optional) TODO: Validate your model.